# Análisis y figuras del caso: vía Google Colab (Python)

Cuaderno guiado para el TFM (estudio de caso). Versión para Colab del flujo del paquete;
en Codex la misma lógica vive en las skills `analizar-datos-tfm` e `interpretar-cambio-individual` con R.

## Antes de subir NADA: privacidad

Este cuaderno se ejecuta en servidores de Google y su asistente procesa lo que subas.
La revisión de privacidad se hace ANTES de cargar el archivo, no después:

- No subas datos que identifiquen o hagan reconocible a la persona estudiada.
- Quitar el nombre o poner "sujeto 1" NO basta: en un estudio de caso, la combinación de
  deporte, club, lesión, edad, fechas o puesto puede delatar a la persona.
- Si tus datos no pueden compartirse, NO uses la vía en la nube: pide el script y ejecútalo
  en tu propio equipo sobre los datos reales.

## Dos vías para interpretar el cambio

- **Con umbral aportado** (celda 6): introduces su valor, unidades, tipo (MDC, MIC, SWC),
  fuente y población/protocolo, y confirmas que aplica a tu caso. El cuaderno dibuja la banda.
- **Sin umbral** (por defecto): seguimiento descriptivo SIN banda. No se afirma que el cambio
  supere el error de medida ni que sea importante; solo se describe la evolución.

Superar el error de medida no demuestra que el cambio sea importante, y ninguna de las dos
cosas prueba que la intervención funcionara.

## 1. Carga de datos

Sube tu CSV con el botón de archivos de Colab (o `Archivo > Subir`) y ajusta `RUTA_DATOS`.
Para practicar, usa `datos-ficticios-caso.csv` (inventado) incluido en el paquete.

Columnas esperadas del ejemplo: `semana`, `fuerza_ext_implicada_nkg`,
`fuerza_ext_contralateral_nkg`, `salto_cmj_cm`, `rpe_medio_0a10`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import sys, os

RUTA_DATOS = "datos-ficticios-caso.csv"   # cambia al nombre de tu archivo
datos = pd.read_csv(RUTA_DATOS)
print(f"Archivo: {RUTA_DATOS} | {len(datos)} filas, {len(datos.columns)} columnas")
datos.head()

## 2. Puerta de datos: estructura, unidades y rangos

Comprueba antes de analizar. Si algo falla, el cuaderno se detiene: corrige el archivo
o ajusta los nombres de columna. No inventes unidades que no consten.

In [ ]:
COLUMNAS = ["semana", "fuerza_ext_implicada_nkg",
            "fuerza_ext_contralateral_nkg", "salto_cmj_cm", "rpe_medio_0a10"]
faltantes = [c for c in COLUMNAS if c not in datos.columns]
assert not faltantes, f"Faltan columnas: {faltantes}"
assert len(datos) > 0, "El archivo no tiene filas"
assert datos["fuerza_ext_implicada_nkg"].between(0, 6).all(), "fuerza implicada fuera de rango (0-6 N/kg)"
assert datos["fuerza_ext_contralateral_nkg"].between(0, 6).all(), "fuerza contralateral fuera de rango (0-6 N/kg)"
assert datos["salto_cmj_cm"].between(0, 100).all(), "CMJ fuera de rango (0-100 cm)"
assert datos["rpe_medio_0a10"].between(0, 10).all(), "RPE fuera de rango (0-10)"
n_na = datos.isna().sum().sum()
print(f"Estructura OK. Valores faltantes: {n_na}")
if n_na:
    print(datos.isna().sum())

## 3. Cambios descriptivos (sin inferencia)

Tabla de cambio primera-última medición y resumen. Son descripciones de ESTA persona:
no son una prueba estadística ni demuestran eficacia.

In [ ]:
datos["lsi_porcentaje"] = (datos["fuerza_ext_implicada_nkg"]
                           / datos["fuerza_ext_contralateral_nkg"] * 100).round(1)

primera, ultima = datos.iloc[0], datos.iloc[-1]
tabla_cambios = pd.DataFrame({
    "variable": ["fuerza_ext_implicada_nkg", "fuerza_ext_contralateral_nkg",
                 "salto_cmj_cm", "lsi_porcentaje"],
    f"semana_{int(primera['semana'])}": [primera["fuerza_ext_implicada_nkg"],
        primera["fuerza_ext_contralateral_nkg"], primera["salto_cmj_cm"], primera["lsi_porcentaje"]],
    f"semana_{int(ultima['semana'])}": [ultima["fuerza_ext_implicada_nkg"],
        ultima["fuerza_ext_contralateral_nkg"], ultima["salto_cmj_cm"], ultima["lsi_porcentaje"]],
})
tabla_cambios["cambio_absoluto"] = (tabla_cambios.iloc[:, 2] - tabla_cambios.iloc[:, 1]).round(2)
tabla_cambios["cambio_porcentaje"] = ((tabla_cambios.iloc[:, 2] / tabla_cambios.iloc[:, 1] - 1) * 100).round(1)
os.makedirs("resultados", exist_ok=True)
tabla_cambios.to_csv("resultados/tabla_cambios.csv", index=False)
tabla_cambios

## 4. Figuras (valores individuales, no solo medias)

Se exportan a `resultados/` en PNG (300 ppp) y PDF, a ancho de columna de revista (~85 mm).
Revisa siempre la figura renderizada: unidades en los ejes, legibilidad y colores
distinguibles también en escala de grises.

In [ ]:
MM_A_PULGADAS = 1 / 25.4
ANCHO, ALTO = 85 * MM_A_PULGADAS, 70 * MM_A_PULGADAS
MARCAR_FICTICIOS = True   # True al practicar con datos-ficticios-caso.csv; pon False con tus datos reales

from matplotlib.ticker import MaxNLocator

def pie_ficticios(fig, texto="Datos ficticios: ejemplo docente"):
    if MARCAR_FICTICIOS:
        fig.get_layout_engine().set(rect=(0, 0.07, 1, 0.93))  # rect=(izq, abajo, ancho, alto): reserva el 7% inferior para el pie
        fig.text(0.99, 0.02, texto, ha="right", va="bottom", fontsize=6, color="grey")

fig, ax = plt.subplots(figsize=(ANCHO, ALTO), layout="constrained")
ax.plot(datos["semana"], datos["fuerza_ext_implicada_nkg"], "o-", label="Implicada")
ax.plot(datos["semana"], datos["fuerza_ext_contralateral_nkg"], "s--", label="Contralateral")
ax.set_xlabel("Semana"); ax.set_ylabel("Fuerza extensores (N/kg)")
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.legend(frameon=False)
pie_ficticios(fig)
fig.savefig("resultados/figura_fuerza.png", dpi=300)
fig.savefig("resultados/figura_fuerza.pdf")
plt.show()

fig, ax = plt.subplots(figsize=(ANCHO, ALTO), layout="constrained")
ax.plot(datos["semana"], datos["salto_cmj_cm"], "o-")
ax.set_xlabel("Semana"); ax.set_ylabel("Altura de salto CMJ (cm)")
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
pie_ficticios(fig)
fig.savefig("resultados/figura_cmj.png", dpi=300)
fig.savefig("resultados/figura_cmj.pdf")
plt.show()


## 5. Umbral de cambio (SOLO si tienes uno y aplica a tu caso)

Si no tienes un umbral, deja `UMBRAL = None` y salta a la celda 6: tu figura queda
descriptiva y está bien así. No inventes un umbral ni uses la variabilidad de tu línea
base para fabricar uno sin un estudio de fiabilidad.

Si tienes uno, rellena TODOS los campos. Avisos:

- `SEM` no se acepta como umbral: el error típico de medida no es por sí solo un umbral
  de cambio (MDC95 = 1,96 x raiz(2) x SEM solo si se cumplen sus supuestos; consúltalo antes).
- `MDC` responde a "¿supera el error de medida?"; `MIC` a "¿es importante?". No son la
  misma banda. Superar el error NO demuestra importancia, y ninguno prueba eficacia.
- La banda se dibuja alrededor de la línea base. En el ejemplo (`SEMANAS_BASELINE = [0]`)
  la línea base es la PRIMERA medición, no una media basal; si defines varias semanas se
  usa su media. Si la fuente no especifica nivel de confianza, llámalo MDC a secas (no
  MDC95) y no llames a la banda "banda de confianza".
  Exige: error aproximadamente constante, línea base estable, misma escala, población y
  protocolo que el estudio de origen del umbral.

In [ ]:
# Vía B (por defecto): sin umbral, seguimiento descriptivo sin banda
UMBRAL = None

# Vía A (ejemplo con los datos ficticios): descomenta y rellena TODO
# UMBRAL = {
#     "valor": 0.20,                 # en las unidades de la variable
#     "tipo": "MDC",                 # "MDC" (supera error) o "MIC" (importante) u otro justificado
#     "fuente": "Autor et al. (año). Revista. DOI",
#     "poblacion_protocolo": "p. ej., adultos readaptación LCA, dinamometría isométrica 90°",
#     "aplicabilidad_confirmada": False,   # pon True solo tras comprobarla de verdad
# }
VARIABLE_UMBRAL = "fuerza_ext_implicada_nkg"
SEMANAS_BASELINE = [0]   # semanas que definen la línea base

if UMBRAL is not None:
    assert UMBRAL["tipo"] != "SEM", "SEM no es un umbral de cambio; revisa la celda de explicación"
    assert UMBRAL.get("aplicabilidad_confirmada") is True, \
        "Confirma primero la aplicabilidad (escala, población, protocolo) poniendo True"
    baseline = datos.loc[datos["semana"].isin(SEMANAS_BASELINE), VARIABLE_UMBRAL].mean()
    fig, ax = plt.subplots(figsize=(ANCHO, ALTO), layout="constrained")
    ax.axhspan(baseline - UMBRAL["valor"], baseline + UMBRAL["valor"], alpha=0.2,
               label=f"{UMBRAL['tipo']} ±{UMBRAL['valor']} sobre línea base")
    ax.axhline(baseline, color="grey", linewidth=0.8)
    ax.plot(datos["semana"], datos[VARIABLE_UMBRAL], "o-")
    ETIQUETAS = {"fuerza_ext_implicada_nkg": "Fuerza extensores (N/kg)",
                 "fuerza_ext_contralateral_nkg": "Fuerza extensores (N/kg)",
                 "salto_cmj_cm": "Altura de salto CMJ (cm)", "rpe_medio_0a10": "RPE (0-10)"}
    ax.set_xlabel("Semana"); ax.set_ylabel(ETIQUETAS.get(VARIABLE_UMBRAL, VARIABLE_UMBRAL))
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.legend(frameon=False, fontsize=7)
    pie_ficticios(fig, "Datos y umbral ficticios: ejemplo docente, no utilizar como referencia.")
    fig.savefig("resultados/figura_umbral.png", dpi=300)
    fig.savefig("resultados/figura_umbral.pdf")
    plt.show()
    _desc_base = f"primera medición (semana {SEMANAS_BASELINE[0]})" if len(SEMANAS_BASELINE) == 1 \
        else f"media de las semanas {SEMANAS_BASELINE}"
    print(f"Línea base = {_desc_base}: {baseline:.2f}. "
          f"Banda {UMBRAL['tipo']}: {baseline - UMBRAL['valor']:.2f} a {baseline + UMBRAL['valor']:.2f}. "
          f"Fuente: {UMBRAL['fuente']}. Recuerda: superar la banda no prueba importancia ni eficacia.")
else:
    print("Sin umbral: seguimiento descriptivo. No se afirma cambio detectable ni importante.")

## 6. Cierre y trazabilidad

- Cada cifra del TFM debe poder señalarse en una salida de este cuaderno (tabla o figura) y
  en la celda que la produce. Una cifra sin esa cadena no entra en el TFM.
- Guarda el cuaderno con sus salidas (Archivo > Descargar .ipynb) junto al CSV usado.
- En el TFM, declara qué herramienta usaste según la normativa del máster.

In [ ]:
import platform
print("Archivos generados:")
for f in sorted(os.listdir("resultados")):
    print(" -", f, os.path.getsize(os.path.join("resultados", f)), "bytes")
print(f"\nPython {platform.python_version()} | pandas {pd.__version__} | matplotlib {plt.matplotlib.__version__}")